# FiiCode 2026 Final — Recursive CatBoost Safe Lags

Goal: create a recursive 14-day forecast.

Current best known public score:
- CatBoost base + historical means: **2.1364**

This notebook predicts test dates one by one:

1. Train CatBoost on full historical train data.
2. For the first test date, build lag/rolling features from real train demand.
3. Predict demand.
4. Insert predicted demand back into the history.
5. Move to the next date.
6. Repeat until all 14 test days are predicted.

This lets future test rows use previous predicted test demand.

## 1. Imports

In [ ]:
import os
import numpy as np
import pandas as pd

from sklearn.metrics import mean_squared_error
from catboost import CatBoostRegressor

## 2. Experiment switches

In [ ]:
CURRENT_BEST_PUBLIC_SCORE = 2.1364

USE_MEAN_FEATURES = True
USE_PRICE_FEATURES = True
USE_LAG_FEATURES = True

# Set this based on the A/B result:
# If rolling=True submission beats rolling=False, keep True.
# If rolling=False is better, set False.
USE_ROLLING_FEATURES = True

USE_LOG_TARGET = False
ROUND_PREDICTIONS = False

RANDOM_SEED = 42

CATBOOST_ITERATIONS = 1500
CATBOOST_LEARNING_RATE = 0.05
CATBOOST_DEPTH = 8
CATBOOST_L2 = 5
CATBOOST_EARLY_STOPPING = 100

SAFE_LAGS = [7, 14, 21, 28]
ROLL_WINDOWS = [7, 14, 28]

OUTPUT_FILE = "submission_catboost_recursive_safe_lags.csv"

## 3. Find input files

In [ ]:
train_path = None
test_path = None

for dirname, _, filenames in os.walk("/kaggle/input"):
    if "train_final.csv" in filenames:
        train_path = os.path.join(dirname, "train_final.csv")
    if "test_final.csv" in filenames:
        test_path = os.path.join(dirname, "test_final.csv")

print("train_path:", train_path)
print("test_path:", test_path)

if train_path is None:
    raise FileNotFoundError("Could not find train_final.csv")
if test_path is None:
    raise FileNotFoundError("Could not find test_final.csv")

## 4. Load data

In [ ]:
train = pd.read_csv(train_path)
test = pd.read_csv(test_path)

train["date"] = pd.to_datetime(train["date"])
test["date"] = pd.to_datetime(test["date"])

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("Train dates:", train["date"].min(), "to", train["date"].max())
print("Test dates:", test["date"].min(), "to", test["date"].max())

display(train.head())
display(test.head())

## 5. Calendar features with shared origin

In [ ]:
ORIGIN_DATE = train["date"].min()

def add_calendar_features(df):
    df = df.copy()
    df["dayofweek"] = df["date"].dt.dayofweek
    df["is_weekend"] = df["dayofweek"].isin([5, 6]).astype(int)
    df["day"] = df["date"].dt.day
    df["month"] = df["date"].dt.month
    df["weekofyear"] = df["date"].dt.isocalendar().week.astype(int)
    df["days_from_start"] = (df["date"] - ORIGIN_DATE).dt.days
    return df

train = add_calendar_features(train)
test = add_calendar_features(test)

## 6. Helper functions: mean features

In [ ]:
def build_mean_tables(source_df):
    tables = {}

    tables["product_store_mean"] = (
        source_df.groupby(["store_id", "product_id"])["demand"]
        .mean()
        .rename("product_store_mean")
        .reset_index()
    )

    tables["product_mean"] = (
        source_df.groupby("product_id")["demand"]
        .mean()
        .rename("product_mean")
        .reset_index()
    )

    tables["store_mean"] = (
        source_df.groupby("store_id")["demand"]
        .mean()
        .rename("store_mean")
        .reset_index()
    )

    tables["category_mean"] = (
        source_df.groupby("category")["demand"]
        .mean()
        .rename("category_mean")
        .reset_index()
    )

    tables["category_store_mean"] = (
        source_df.groupby(["store_id", "category"])["demand"]
        .mean()
        .rename("category_store_mean")
        .reset_index()
    )

    tables["product_weekday_mean"] = (
        source_df.groupby(["product_id", "dayofweek"])["demand"]
        .mean()
        .rename("product_weekday_mean")
        .reset_index()
    )

    tables["product_store_weekday_mean"] = (
        source_df.groupby(["store_id", "product_id", "dayofweek"])["demand"]
        .mean()
        .rename("product_store_weekday_mean")
        .reset_index()
    )

    return tables


def add_mean_features(df, tables, global_mean):
    df = df.copy()

    df = df.merge(tables["product_store_mean"], on=["store_id", "product_id"], how="left")
    df = df.merge(tables["product_mean"], on="product_id", how="left")
    df = df.merge(tables["store_mean"], on="store_id", how="left")
    df = df.merge(tables["category_mean"], on="category", how="left")
    df = df.merge(tables["category_store_mean"], on=["store_id", "category"], how="left")
    df = df.merge(tables["product_weekday_mean"], on=["product_id", "dayofweek"], how="left")
    df = df.merge(tables["product_store_weekday_mean"], on=["store_id", "product_id", "dayofweek"], how="left")

    mean_cols = [
        "product_store_mean",
        "product_mean",
        "store_mean",
        "category_mean",
        "category_store_mean",
        "product_weekday_mean",
        "product_store_weekday_mean"
    ]

    for c in mean_cols:
        df[c] = df[c].fillna(global_mean)

    return df

## 7. Helper functions: price features

In [ ]:
def build_price_tables(source_df):
    return {
        "product_avg_price": (
            source_df.groupby("product_id")["price"]
            .mean()
            .rename("product_avg_price")
            .reset_index()
        ),
        "category_avg_price": (
            source_df.groupby("category")["price"]
            .mean()
            .rename("category_avg_price")
            .reset_index()
        )
    }


def add_price_features(df, tables):
    df = df.copy()

    df = df.merge(tables["product_avg_price"], on="product_id", how="left")
    df = df.merge(tables["category_avg_price"], on="category", how="left")

    df["product_avg_price"] = df["product_avg_price"].fillna(df["price"])
    df["category_avg_price"] = df["category_avg_price"].fillna(df["price"])

    df["price_vs_product_avg"] = df["price"] / df["product_avg_price"].replace(0, np.nan)
    df["price_vs_category_avg"] = df["price"] / df["category_avg_price"].replace(0, np.nan)
    df["price_diff_product_avg"] = df["price"] - df["product_avg_price"]
    df["price_diff_category_avg"] = df["price"] - df["category_avg_price"]

    for c in [
        "price_vs_product_avg",
        "price_vs_category_avg",
        "price_diff_product_avg",
        "price_diff_category_avg"
    ]:
        df[c] = df[c].replace([np.inf, -np.inf], np.nan).fillna(0)

    return df

## 8. Helper functions: lag and rolling features

In [ ]:
def add_lag_rolling_features(df, group_cols=["store_id", "product_id"]):
    df = df.copy()
    df = df.sort_values(group_cols + ["date"]).reset_index(drop=True)

    lag_cols = []
    roll_cols = []

    if USE_LAG_FEATURES:
        for lag in SAFE_LAGS:
            col = f"lag_{lag}"
            df[col] = df.groupby(group_cols)["demand"].shift(lag)
            lag_cols.append(col)

    if USE_ROLLING_FEATURES:
        for window in ROLL_WINDOWS:
            col = f"roll_mean_{window}"
            df[col] = (
                df.groupby(group_cols)["demand"]
                .transform(lambda s: s.shift(1).rolling(window, min_periods=1).mean())
            )
            roll_cols.append(col)

    return df, lag_cols, roll_cols


def fill_lag_roll_missing(df, lag_cols, roll_cols, global_mean):
    df = df.copy()
    for c in lag_cols + roll_cols:
        df[c] = df[c].fillna(df.get("product_store_mean"))
        df[c] = df[c].fillna(df.get("product_mean"))
        df[c] = df[c].fillna(df.get("category_store_mean"))
        df[c] = df[c].fillna(df.get("category_mean"))
        df[c] = df[c].fillna(global_mean)
    return df

## 9. Validation check: recursive-style last 14 days

In [ ]:
last_train_date = train["date"].max()
valid_start = last_train_date - pd.Timedelta(days=13)

tr_raw = train[train["date"] < valid_start].copy()
va_raw = train[train["date"] >= valid_start].copy()

print("Training period:", tr_raw["date"].min(), "to", tr_raw["date"].max())
print("Validation period:", va_raw["date"].min(), "to", va_raw["date"].max())
print("tr_raw:", tr_raw.shape, "va_raw:", va_raw.shape)

## 10. Build training features for validation model

Training features are built from known historical demand only.

In [ ]:
tr_global_mean = tr_raw["demand"].mean()

tr_features_df = tr_raw.copy()
tr_features_df["part"] = "train"

if USE_MEAN_FEATURES:
    tr_mean_tables = build_mean_tables(tr_raw)
    tr_features_df = add_mean_features(tr_features_df, tr_mean_tables, tr_global_mean)

if USE_PRICE_FEATURES:
    tr_price_tables = build_price_tables(tr_raw)
    tr_features_df = add_price_features(tr_features_df, tr_price_tables)

tr_features_df, lag_cols, roll_cols = add_lag_rolling_features(tr_features_df)
tr_features_df = fill_lag_roll_missing(tr_features_df, lag_cols, roll_cols, tr_global_mean)

print("lag cols:", lag_cols)
print("roll cols:", roll_cols)
display(tr_features_df.head())

## 11. Prepare model columns

In [ ]:
cat_cols = [
    "store_id",
    "city",
    "region",
    "product_id",
    "product_name",
    "category",
    "holiday_name"
]
cat_cols = [c for c in cat_cols if c in tr_features_df.columns]

drop_cols = [
    "row_ID",
    "row_id",
    "date",
    "demand",
    "part"
]

for c in cat_cols:
    tr_features_df[c] = tr_features_df[c].fillna("missing").astype(str)

features = [c for c in tr_features_df.columns if c not in drop_cols]
cat_feature_indices = [features.index(c) for c in cat_cols if c in features]

print("Number of features:", len(features))
print(features)
print("Categorical feature indices:", cat_feature_indices)

## 12. Train validation model

In [ ]:
X_tr = tr_features_df[features]
y_tr = tr_features_df["demand"]

if USE_LOG_TARGET:
    y_train_model = np.log1p(y_tr)
else:
    y_train_model = y_tr

model = CatBoostRegressor(
    loss_function="RMSE",
    eval_metric="RMSE",
    iterations=CATBOOST_ITERATIONS,
    learning_rate=CATBOOST_LEARNING_RATE,
    depth=CATBOOST_DEPTH,
    l2_leaf_reg=CATBOOST_L2,
    random_seed=RANDOM_SEED,
    verbose=100
)

model.fit(
    X_tr,
    y_train_model,
    cat_features=cat_feature_indices
)

## 13. Recursive validation prediction

In [ ]:
history = tr_raw.copy()
validation_predictions = []

for current_date in sorted(va_raw["date"].unique()):
    current_rows = va_raw[va_raw["date"] == current_date].copy()
    current_rows["demand"] = np.nan
    current_rows["part"] = "valid_current"

    temp = pd.concat([history, current_rows], axis=0, ignore_index=True)

    if USE_MEAN_FEATURES:
        temp = add_mean_features(temp, tr_mean_tables, tr_global_mean)

    if USE_PRICE_FEATURES:
        temp = add_price_features(temp, tr_price_tables)

    temp, cur_lag_cols, cur_roll_cols = add_lag_rolling_features(temp)
    temp = fill_lag_roll_missing(temp, cur_lag_cols, cur_roll_cols, tr_global_mean)

    current_fe = temp[temp["part"] == "valid_current"].copy()

    for c in cat_cols:
        current_fe[c] = current_fe[c].fillna("missing").astype(str)

    pred = model.predict(current_fe[features])

    if USE_LOG_TARGET:
        pred = np.expm1(pred)

    pred = np.clip(pred, 0, None)

    current_output = current_fe[["row_id", "date", "store_id", "product_id"]].copy()
    current_output["prediction"] = pred
    validation_predictions.append(current_output)

    # Insert predictions into history as if they were known demand.
    history_add = va_raw[va_raw["date"] == current_date].copy()
    history_add["demand"] = pred
    history = pd.concat([history, history_add], axis=0, ignore_index=True)

valid_pred_df = pd.concat(validation_predictions, axis=0, ignore_index=True)

valid_eval = va_raw[["row_id", "demand"]].merge(valid_pred_df[["row_id", "prediction"]], on="row_id", how="left")
recursive_rmse = mean_squared_error(valid_eval["demand"], valid_eval["prediction"], squared=False)

print("Recursive validation RMSE:", recursive_rmse)
display(valid_eval.head())
display(valid_eval["prediction"].describe())

## 14. Feature importance

In [ ]:
importance = pd.DataFrame({
    "feature": features,
    "importance": model.get_feature_importance()
}).sort_values("importance", ascending=False)

display(importance.head(40))

## 15. Train final model on full train

In [ ]:
full_global_mean = train["demand"].mean()

full_train_features = train.copy()
full_train_features["part"] = "train"

if USE_MEAN_FEATURES:
    full_mean_tables = build_mean_tables(train)
    full_train_features = add_mean_features(full_train_features, full_mean_tables, full_global_mean)

if USE_PRICE_FEATURES:
    full_price_tables = build_price_tables(train)
    full_train_features = add_price_features(full_train_features, full_price_tables)

full_train_features, final_lag_cols, final_roll_cols = add_lag_rolling_features(full_train_features)
full_train_features = fill_lag_roll_missing(full_train_features, final_lag_cols, final_roll_cols, full_global_mean)

for c in cat_cols:
    full_train_features[c] = full_train_features[c].fillna("missing").astype(str)

features_final = [c for c in full_train_features.columns if c not in drop_cols]
cat_feature_indices_final = [features_final.index(c) for c in cat_cols if c in features_final]

X_full = full_train_features[features_final]
y_full = full_train_features["demand"]

if USE_LOG_TARGET:
    y_full_model = np.log1p(y_full)
else:
    y_full_model = y_full

final_model = CatBoostRegressor(
    loss_function="RMSE",
    eval_metric="RMSE",
    iterations=CATBOOST_ITERATIONS,
    learning_rate=CATBOOST_LEARNING_RATE,
    depth=CATBOOST_DEPTH,
    l2_leaf_reg=CATBOOST_L2,
    random_seed=RANDOM_SEED,
    verbose=100
)

final_model.fit(
    X_full,
    y_full_model,
    cat_features=cat_feature_indices_final
)

## 16. Recursive test prediction

In [ ]:
history = train.copy()
test_predictions_parts = []

for current_date in sorted(test["date"].unique()):
    current_rows = test[test["date"] == current_date].copy()
    current_rows["demand"] = np.nan
    current_rows["part"] = "test_current"

    temp = pd.concat([history, current_rows], axis=0, ignore_index=True)

    if USE_MEAN_FEATURES:
        temp = add_mean_features(temp, full_mean_tables, full_global_mean)

    if USE_PRICE_FEATURES:
        temp = add_price_features(temp, full_price_tables)

    temp, cur_lag_cols, cur_roll_cols = add_lag_rolling_features(temp)
    temp = fill_lag_roll_missing(temp, cur_lag_cols, cur_roll_cols, full_global_mean)

    current_fe = temp[temp["part"] == "test_current"].copy()

    for c in cat_cols:
        current_fe[c] = current_fe[c].fillna("missing").astype(str)

    pred = final_model.predict(current_fe[features_final])

    if USE_LOG_TARGET:
        pred = np.expm1(pred)

    pred = np.clip(pred, 0, None)

    if ROUND_PREDICTIONS:
        pred = np.rint(pred).astype(int)

    out = current_fe[["row_id", "date", "store_id", "product_id"]].copy()
    out["demand"] = pred
    test_predictions_parts.append(out)

    # Insert predicted demand back into history.
    history_add = test[test["date"] == current_date].copy()
    history_add["demand"] = pred
    history = pd.concat([history, history_add], axis=0, ignore_index=True)

test_pred_df = pd.concat(test_predictions_parts, axis=0, ignore_index=True)

print(test_pred_df.shape)
display(test_pred_df.head())
display(test_pred_df["demand"].describe())

## 17. Create submission

In [ ]:
submission = test_pred_df[["row_id", "demand"]].copy()
submission = submission.sort_values("row_id").reset_index(drop=True)

submission.to_csv(OUTPUT_FILE, index=False)

display(submission.head())
print("Saved:", OUTPUT_FILE)

## 18. How to use this notebook

Run this if:

- rolling=True safe lag beats rolling=False; keep `USE_ROLLING_FEATURES = True`;
- rolling=False beats rolling=True; set `USE_ROLLING_FEATURES = False`.

Submit:

```text
submission_catboost_recursive_safe_lags.csv
```

If this beats `2.1364`, it becomes the new anchor.